# 07 · Phase 5: evaluating ungrounded vs. grounded vs. verified explanations

The main experiment. Phases 3 and 4 were tuned on a 48-image pilot (output limits, citation parsing, when the
verifier asks for a revision), so this notebook runs the **frozen** pipeline on a **fresh test sample**: test videos
that the pilot and the worked examples did not use. Each image is explained by the same three Groq models in all
three conditions, so every comparison is **paired by image**.

| Condition | What the LLM gets |
|---|---|
| ungrounded | phase 3: the evidence summary only |
| grounded | phase 4: the summary + retrieved evidence cards, which it must cite |
| verified | the grounded answer, checked; one revision request if a check fails |
| *baselines* | no LLM: fixed rules scored with the same checks (see below) |

**Pre-registered primary hypotheses** (per model, grounded vs. ungrounded, two-sided, Holm-corrected over all
12 tests, α = 0.05):
1. grounding raises the share of sentences supported by the evidence (`supported_share`, Wilcoxon)
2. grounding lowers the share of contradicted sentences (`contradicted_share`, Wilcoxon)
3. grounding changes how often the artefact fits the true manipulation (`method_consistent`, McNemar)
4. grounding lowers GAN-fingerprint claims on images that are not GAN-generated (`gan_claim`, McNemar)

Everything else (verified vs. grounded, the other metrics, the baselines, agreement between models,
breakdowns) is secondary and reported with its own Holm correction.

**Metrics**, in five groups:
- *output*: valid JSON with allowed labels
- *heatmap faithfulness*: region in the top 3, pointing hit, region IoU
- *artefact*: consistent with the verdict, plausible for the regions, consistent with the true method, GAN claims
- *attribution*: NLI-supported and contradicted sentences, valid citations, artefact and region backed by a cited card
- *language*: hedging on uncertain verdicts, certainty words, and describing the image as if seen. These are
  keyword checks, so they are approximate.

**Baselines** answer without an LLM:
- *top region + first card*: the evidence pipeline alone
- *always mouth*: the most common pilot answer
- *random plausible*: the expected score of a random region and artefact that fit the heatmap

A baseline shows whether a rate means anything. For example, a pointing hit of 65% is not impressive if naming
the top region scores 70%.

**Batches.** The test set (`N_PER_DATASET` per dataset, 96 images by default) is split into `N_BATCHES`
balanced batches. A batch uses about 140K Groq tokens per model, under the 200K daily limit, so run **one batch
per day**:
1. **Day 1:** `BATCH = 0`.
2. **Next day:** `BATCH = 1`, with the previous version's output attached.
3. **Day after:** `BATCH = 2`, with that version's output attached.

Every run copies the earlier batches into its own output and analyses all batches found. `BATCH = None` only
re-runs the analysis (no GPU or API key needed).

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 (detector, embeddings, NLI) · **Internet:** On
- **Inputs:** **01_extract_faces**, **01b_extract_celebdf**, **02_train** (the copy with EfficientNet-B0),
  **05_explanations** (its output lists the pilot images to exclude), and from batch 1 on the previous saved output
  of this notebook
- **Secrets:** `GROQ_API_KEY`

**Time:** about 20 minutes per GPT-OSS model and 75 minutes for Qwen (60 s spacing) per batch, plus a few
minutes for the detector and the NLI model.

**Produces:** `/kaggle/working/outputs/evaluation/`:
- every answer with its raw reply, per batch
- per-answer scores and sentence NLI labels
- summary tables, all statistical tests, the agreement table and a forest plot
- a sentence sample for checking the NLI model by hand

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam openai
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
%matplotlib inline
import json, os, shutil, time
from dataclasses import fields
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.kaggle_utils import setup, find, find_all, find_datasets, checkpoint_roots, setup_llm
setup(data=False)

from src.eval import experiment as ex
assert hasattr(ex, "score"), "Outdated code: restart the session (Run → Restart) and run all again."
from src.classifier.compare import find_checkpoints
from src.eval.experiment import BASELINE_LLM, BASELINES, CONTINUOUS, HIGHER_IS_BETTER, METRICS, PRIMARY
from src.eval.stats import add_holm, compare, kappa_ci, mean_ci
from src.eval.verify import NLI_LABELS, EvidenceVerifier, NLIChecker
from src.explain.corpus import load_corpus
from src.explain.llm import ForensicExplainer, LLMExplanation, normalise_citation
from src.explain.retriever import Retriever
from src.xai.gradcam import Explainer

DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
CKPTS = find_checkpoints(checkpoint_roots())
P3 = find("evidence.csv")
OUT = Path("/kaggle/working/outputs/evaluation")
OUT.mkdir(parents=True, exist_ok=True)
print("datasets:", DATASETS)
print("detectors available:", list(CKPTS))
print("phase 3 evidence table (pilot images to exclude):", P3)
cards = load_corpus()
print(f"corpus: {len(cards)} cards; {sum(not c.checked for c in cards)} not yet checked against their source")

## Settings

In [ ]:
DETECTOR = "efficientnet_b0"                  # chosen in notebook 04b
LLMS = [("groq", "openai/gpt-oss-120b"),      # same three models and settings as phases 3 and 4
        ("groq", "openai/gpt-oss-20b"),
        ("groq", "qwen/qwen3.8-27b")]
MIN_INTERVAL = {"groq": 15}                   # seconds between requests (free-tier per-minute token limit)
MODEL_LIMITS = {"qwen/qwen3.8-27b": {"min_interval": 60, "max_tokens": 1000}}  # Groq: 1K output tokens/min for Qwen
os.environ["LLM_REASONING_EFFORT"] = "low"    # same reasoning effort for every model
RANK_BY = "concentration"                     # as in phases 3 and 4
RANKERS = ("region", "bm25", "dense")         # phase 4's retriever
REVISE_ON = ("contradiction",)                # phase 4's verifier
N_PER_DATASET = 48                            # held-out test images per dataset (one frame per unused test video)
N_BATCHES = 3                                 # the test set is run in this many daily batches
BATCH = 0                                     # this run's batch: 0, 1, ..., N_BATCHES - 1 (one per day);
                                              # None = analysis only, from attached outputs of this notebook
SEED = 1                                      # test-sample seed (the pilot used 0); keep it fixed across batches
RUN_MODELS = None                             # None = every model; or a subset to re-run this batch, loading the
                                              # others from an attached earlier run of the same batch
ALPHA = 0.05
assert RUN_MODELS is None or set(RUN_MODELS) <= {m for _, m in LLMS}, "RUN_MODELS must be models from LLMS"
if BATCH is not None:
    assert 0 <= BATCH < N_BATCHES, "BATCH must be between 0 and N_BATCHES - 1"
    assert DETECTOR in CKPTS, f"{DETECTOR} not found: attach the 02_train copy that trained it"
    assert P3 is not None, "Attach the saved output of 05_explanations (its pilot images are excluded)"

## 1 · Held-out test images
One frame per test video, balanced across manipulation methods, from videos not used by the pilot or the worked
examples. The full sample is fixed by `SEED`; each batch takes every `N_BATCHES`-th image of it, so every batch
has the same mix of datasets and methods. The detector and Grad-CAM++ run once per image.

In [ ]:
items = []
if BATCH is not None:
    pilot = pd.read_csv(P3)
    sample = pd.concat([ex.test_sample(Path(root), N_PER_DATASET,
                                       set(pilot[pilot.dataset == ds].file.map(ex.video_of)), SEED).assign(dataset=ds)
                        for ds, root in DATASETS.items()])
    sample = sample.sort_values(["dataset", "method", "file"]).reset_index(drop=True)
    sample["batch"] = sample.index % N_BATCHES
    for ds in DATASETS:
        assert not set(sample[sample.dataset == ds].video) & set(pilot[pilot.dataset == ds].file.map(ex.video_of))
    display(sample.pivot_table(index=["dataset", "method"], columns="batch", values="file", aggfunc="count"))

    explainer = Explainer(CKPTS[DETECTOR], rank_by=RANK_BY)
    for r in sample[sample.batch == BATCH].itertuples():
        items.append({"batch": BATCH, "dataset": r.dataset, "file": r.file, "path": r.path,
                      "manipulation": r.method, "label": r.label, "exp": explainer(Image.open(r.path))})
    del explainer
    evidence = pd.DataFrame([{k: v for k, v in it.items() if k not in ("exp", "path")} | {
        "pred": it["exp"].label, "prob_fake": round(it["exp"].prob_fake, 4), "confidence": it["exp"].confidence,
        "top_regions": [r for r, _ in it["exp"].top_regions[:3]], "summary": it["exp"].describe()} for it in items])
    evidence.to_csv(OUT / f"evidence_b{BATCH}.csv", index=False)
    print(f"batch {BATCH}: {len(items)} images; detector accuracy {(evidence.pred == evidence.label).mean():.0%}; "
          f"confidence: {evidence.confidence.value_counts().to_dict()}")
else:
    print("BATCH = None: analysis only")

## 2 · Run each LLM: ungrounded, grounded, verified
Same prompts as phases 3 and 4: **ungrounded** is phase 3's prompt, **grounded** adds the retrieved cards and
the instruction to cite them, and **verified** checks each grounded answer and asks once for a revision when a
check fails. A failed request (e.g. a rate limit) is kept as an error row and left out of every metric.

In [ ]:
FIELDS = {f.name for f in fields(LLMExplanation)}
CONDITIONS = ("ungrounded", "grounded", "verified")


def load_answers(path):
    # Saved answers of this batch, in the order of `items`, with citation ids normalised.
    by_key = {(r["dataset"], r["file"]): r for r in map(json.loads, open(path))}
    outs = [LLMExplanation(**{k: v for k, v in by_key[(it["dataset"], it["file"])].items() if k in FIELDS})
            for it in items]
    for o in outs:
        o.citations = [c for c in map(normalise_citation, o.citations) if c]
    return outs


results, run_log = {}, []
if BATCH is not None:
    retriever = Retriever(RANKERS)
    verifier = EvidenceVerifier(NLIChecker(), revise_on=REVISE_ON)
    exps = [it["exp"] for it in items]
    for provider, model_id in LLMS:
        name = model_id.replace("/", "__")
        print(f"\n===== {provider}: {model_id} =====")
        t0 = time.time()
        if RUN_MODELS is not None and model_id not in RUN_MODELS:
            paths = {c: find(f"phase5_{c}_{name}_b{BATCH}.jsonl") for c in CONDITIONS}
            if None in paths.values():
                print(f"SKIPPED {model_id}: no earlier run of batch {BATCH} attached; add the model to RUN_MODELS")
                run_log.append({"llm": model_id, "status": f"no earlier batch {BATCH} output found"})
                continue
            for c, p in paths.items():
                results[(c, model_id)] = load_answers(p)
                shutil.copy(p, OUT / p.name)
            run_log.append({"llm": model_id, "status": f"loaded from {paths['grounded'].parent}"})
            continue
        limits = MODEL_LIMITS.get(model_id, {})
        os.environ["LLM_MIN_INTERVAL"] = str(limits.get("min_interval", MIN_INTERVAL.get(provider, 0)))
        if "max_tokens" in limits:
            os.environ["LLM_MAX_TOKENS"] = str(limits["max_tokens"])
        else:
            os.environ.pop("LLM_MAX_TOKENS", None)    # the provider's default (1024 on Groq)
        try:
            if not setup_llm(provider, model_id):
                raise RuntimeError("add the GROQ_API_KEY secret")
            llm = ForensicExplainer(retriever=retriever)
            conditions = {"ungrounded": llm.explain_batch(exps, grounded=False)}
            conditions["grounded"] = llm.explain_batch(exps, grounded=True)
            conditions["verified"] = llm.revise_batch(exps, conditions["grounded"], verifier)
            for c, outs in conditions.items():
                results[(c, model_id)] = outs
                rows = [{"llm": model_id, "provider": provider, "retriever": retriever.describe(),
                         **{k: v for k, v in it.items() if k not in ("exp", "path")}, **o.to_dict()}
                        for it, o in zip(items, outs)]
                with open(OUT / f"phase5_{c}_{name}_b{BATCH}.jsonl", "w") as f:
                    f.writelines(json.dumps(r) + "\n" for r in rows)
            log = {"llm": model_id, "status": "done", "minutes": round((time.time() - t0) / 60, 1)}
            log.update({f"{c}_valid": f"{np.mean([o.schema_ok for o in outs]):.0%}" for c, outs in conditions.items()})
            log["request_errors"] = sum(bool(o.error) and not o.raw for outs in conditions.values() for o in outs)
            log["revised"] = sum(o.revised for o in conditions["verified"])
            run_log.append(log)
            del llm
        except Exception as e:
            print(f"SKIPPED {model_id}: {type(e).__name__}: {e}")
            run_log.append({"llm": model_id, "status": f"failed: {type(e).__name__}: {e}"[:120],
                            "minutes": round((time.time() - t0) / 60, 1)})
    display(pd.DataFrame(run_log))
    first_error = next((o.error for outs in results.values() for o in outs if o.error and not o.raw), "")
    if first_error:
        print("first request error:", first_error[:300])
    assert results, "No LLM produced explanations; see the errors above."

## 3 · Score every answer and the baselines
This batch is scored here (the heatmap checks need the detector's heatmaps); earlier batches are read from their
saved scores and copied into this output, so the latest version always holds every batch.

In [ ]:
if BATCH is not None:
    scores_b, sentences_b = ex.score(items, results, verifier, retriever)
    scores_b.to_csv(OUT / f"phase5_scores_b{BATCH}.csv", index=False)
    sentences_b.to_csv(OUT / f"phase5_sentences_b{BATCH}.csv", index=False)
    print(f"batch {BATCH}: {len(scores_b)} scored answers, {len(sentences_b)} sentences")

earlier = {}
for p in find_all("phase5_scores_b*.csv"):
    b = int(p.stem.rsplit("_b", 1)[1])
    if b != BATCH and b not in earlier:
        earlier[b] = p
for b, p in sorted(earlier.items()):
    for f in p.parent.glob(f"*_b{b}.*"):
        shutil.copy(f, OUT / f.name)
batches = sorted(set(earlier) | ({BATCH} if BATCH is not None else set()))
assert batches, "No scores: run a batch, or attach an earlier output of this notebook"
scores = pd.concat([pd.read_csv(OUT / f"phase5_scores_b{b}.csv") for b in batches], ignore_index=True)
sentences = pd.concat([pd.read_csv(OUT / f"phase5_sentences_b{b}.csv") for b in batches
                       if (OUT / f"phase5_sentences_b{b}.csv").exists()], ignore_index=True)
M = [m for m in METRICS if m in scores]
scores[M] = scores[M].astype(float)
scores["request_failed"] = scores.request_failed.astype(str).str.lower().eq("true")
ok = scores[~scores.request_failed]
MODELS = sorted(set(ok.llm) - {BASELINE_LLM})
ORDER = [c for c in ("ungrounded", "grounded", "verified", *BASELINES) if c in set(scores.condition)]
missing = sorted(set(range(N_BATCHES)) - set(batches))
print(f"batches: {batches}; {ok.image.nunique()} images; models: {MODELS}"
      + (f"\nPRELIMINARY: batches {missing} not run yet" if missing else " (complete test set)"))
failed = scores[scores.request_failed].groupby(["llm", "condition"]).size()
if len(failed):
    print("failed requests (left out of every metric):", failed.to_dict())

## 4 · Results by condition
Mean and 95% bootstrap interval over images. Rates are shares of answers; a check that does not apply to an
answer (e.g. method consistency on a REAL verdict, citation checks without cards) is left out of its rate.
Lower is better for `gan_claim`, `contradicted_share`, `overclaims` and `visual_claim`.

In [ ]:
def ci_table(df):
    t = df.groupby("condition")[M].agg(mean_ci).T
    t = t[[c for c in ORDER if c in t]]
    return t[(t != "").any(axis=1)].replace("", "—")


display(pd.DataFrame({m: {"group": g, "higher is better": h, "meaning": d} for m, (g, h, d) in METRICS.items()
                      if m in M}).T)
for llm in MODELS:
    display(Markdown(f"**{llm}** ({ok[ok.llm == llm].image.nunique()} images)"))
    display(ci_table(ok[ok.llm == llm]))
display(Markdown("**No-LLM baselines** (no text, so the attribution and language checks do not apply)"))
display(ci_table(ok[ok.llm == BASELINE_LLM]))
summary = ok.groupby(["llm", "condition"])[M].mean()
summary = summary.reindex([(l, c) for l in MODELS + [BASELINE_LLM] for c in ORDER if (l, c) in summary.index])
summary.round(4).to_csv(OUT / "phase5_summary.csv")
latency = ok[ok.llm != BASELINE_LLM].groupby(["llm", "condition"]).agg(
    seconds=("latency_s", "mean"), revised=("revised", lambda x: x.astype(str).str.lower().eq("true").mean()))
display(Markdown("**Cost:** mean seconds per answer and share of answers revised"))
display(latency.round(2))

## 5 · Primary hypotheses: grounded vs. ungrounded
Paired by image. `diff` = grounded − ungrounded, with a 95% bootstrap interval. McNemar's exact test for yes/no
checks (`effect`: images that changed in each direction), Wilcoxon signed-rank for shares (`effect`: rank-biserial
r). `p_holm` is corrected over the 12 primary tests. Only pairs where both answers have the check count (`n`).

In [ ]:
COLS = ["llm", "comparison", "metric", "n", "before", "after", "diff", "ci", "test", "effect", "p", "p_holm", "verdict"]


def show(tests, name):
    if not len(tests):
        print("no paired data"); return
    t = tests.assign(ci=tests.apply(lambda r: f"[{r.ci_low:+.3f}, {r.ci_high:+.3f}]", axis=1))
    t.to_csv(OUT / f"phase5_{name}.csv", index=False)
    display(t[COLS].round({"before": 3, "after": 3, "diff": 3, "p": 4, "p_holm": 4}))


primary = add_holm(compare(ok, [("ungrounded", "grounded")], PRIMARY, CONTINUOUS), ALPHA, HIGHER_IS_BETTER)
show(primary, "primary_tests")

## 6 · Secondary comparisons
- **Grounded vs. ungrounded, other metrics**, and **verified vs. grounded, all metrics**: one Holm family.
- **LLM vs. baselines** on the heatmap and artefact checks: another family. *Top region + first card* asks
  whether the LLM adds anything to the evidence pipeline; *random plausible* asks whether it beats chance among
  answers that fit the heatmap.

In [ ]:
secondary = pd.concat([compare(ok, [("ungrounded", "grounded")], [m for m in M if m not in PRIMARY], CONTINUOUS),
                       compare(ok, [("grounded", "verified")], M, CONTINUOUS)], ignore_index=True)
show(add_holm(secondary, ALPHA, HIGHER_IS_BETTER), "secondary_tests")

CHECKS = [m for m in ["region_in_top3", "pointing_hit", "region_iou", "artifact_plausible", "method_consistent",
                      "gan_claim"] if m in M]
pairs = [(b, c) for b in ("top region + first card", "random plausible") for c in ("ungrounded", "grounded", "verified")]
display(Markdown("**LLM vs. no-LLM baselines** (`before` = baseline, `after` = LLM)"))
show(add_holm(compare(ok, pairs, CHECKS, CONTINUOUS), ALPHA, HIGHER_IS_BETTER), "baseline_tests")

## 7 · Do the models agree on the artefact type?
Fleiss' κ over FAKE verdicts, with a 95% bootstrap interval, on the images every model answered validly in every
condition (so the conditions are compared on the same images). κ near 0 means the choice is close to a guess; in
the pilot it was 0.16 ungrounded.

In [ ]:
wide = {c: ok[(ok.condition == c) & (ok.verdict == "fake") & (ok.llm != BASELINE_LLM)].pivot_table(
            index="image", columns="llm", values="artifact", aggfunc="first").dropna() for c in CONDITIONS}
wide = {c: w for c, w in wide.items() if w.shape[1] >= 2}
common = sorted(set.intersection(*(set(w.index) for w in wide.values()))) if wide else []
agreement = []
for c, w in wide.items():
    w = w.loc[common]
    if len(w) >= 2:
        k, lo, hi = kappa_ci(w)
        agreement.append({"condition": c, "fake images": len(w), "Fleiss kappa": round(k, 3),
                          "95% CI": f"[{lo:.3f}, {hi:.3f}]", "all models agree": round(float((w.nunique(axis=1) == 1).mean()), 3),
                          "distinct artefacts used": int(pd.unique(w.values.ravel()).size)})
agreement = pd.DataFrame(agreement)
agreement.to_csv(OUT / "phase5_agreement.csv", index=False)
display(agreement)

## 8 · Breakdowns
Means by dataset (cross-dataset: the detector was trained on FF++ only), by whether the detector's verdict was
right (a wrong verdict is still explained, and the explanation should not invent support for it), and by
confidence level.

In [ ]:
KEY = [m for m in ["supported_share", "contradicted_share", "method_consistent", "pointing_hit", "hedges_when_uncertain"]
       if m in M]
llm_rows = ok[ok.llm != BASELINE_LLM]
for factor in ("dataset", "verdict_correct", "confidence"):
    display(Markdown(f"**By {factor}** (all three models)"))
    t = llm_rows.groupby([factor, "condition"])[KEY].mean().unstack("condition").round(3)
    t.columns = [f"{m} · {c}" for m, c in t.columns]
    display(t[[f"{m} · {c}" for m in KEY for c in CONDITIONS if f"{m} · {c}" in t]])
    display(llm_rows.groupby(factor).image.nunique().rename("images").to_frame().T)

In [ ]:
tests = pd.concat([primary, secondary], ignore_index=True)
FOREST = [m for m in PRIMARY + ["pointing_hit", "valid", "hedges_when_uncertain"] if m in set(tests.metric)]
colors = {"ungrounded → grounded": "#2a6f97", "grounded → verified": "#c8553d"}
fig, axes = plt.subplots(1, len(FOREST), figsize=(2.6 * len(FOREST), 0.5 + 0.55 * len(MODELS)), sharey=True)
for ax, m in zip(np.atleast_1d(axes), FOREST):
    for j, (comp, color) in enumerate(colors.items()):
        t = tests[(tests.metric == m) & (tests.comparison == comp)].set_index("llm").reindex(MODELS)
        y = np.arange(len(MODELS)) + (j - 0.5) * 0.25
        ax.errorbar(t["diff"], y, xerr=[t["diff"] - t.ci_low, t.ci_high - t["diff"]], fmt="o", color=color,
                    ms=4, capsize=2, label=comp)
    ax.axvline(0, color="#888", lw=0.8)
    ax.set_title(f"{m}\n({'higher' if HIGHER_IS_BETTER[m] else 'lower'} is better)", fontsize=8)
    ax.tick_params(labelsize=7)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
np.atleast_1d(axes)[0].set_yticks(range(len(MODELS)), MODELS)
np.atleast_1d(axes)[-1].legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
fig.suptitle("Paired differences with 95% bootstrap intervals", fontsize=9, x=0.01, ha="left")
plt.tight_layout(); plt.savefig(OUT / "phase5_differences.png", dpi=150, bbox_inches="tight"); plt.show()

## 9 · Checking the NLI model
The attribution metrics depend on an NLI model, which measures whether a sentence follows from the evidence and
the cited cards, not whether it is true. This cell exports a sample stratified by condition and NLI label: fill
`your_label` (entailment, neutral or contradiction) and attach the file as a dataset. Any attached
`nli_check_sample*.csv` with labels (including phase 4's pilot sample) is compared with the model here.

In [ ]:
from sklearn.metrics import cohen_kappa_score

if len(sentences):
    check = sentences.sample(frac=1, random_state=0).groupby(["condition", "nli_label"]).head(15)
    check.assign(your_label="").to_csv(OUT / "nli_check_sample_test.csv", index=False)
    print(f"exported {len(check)} sentences to nli_check_sample_test.csv")

labelled = []
for p in find_all("nli_check_sample*.csv"):
    d = pd.read_csv(p)
    if "your_label" in d:
        d["your_label"] = d.your_label.astype(str).str.strip().str.lower()
        labelled.append(d[d.your_label.isin(NLI_LABELS)].assign(source=p.name))
labelled = [d for d in labelled if len(d)]
if labelled:
    lab = pd.concat(labelled, ignore_index=True).drop_duplicates(["sentence", "premise"])
    entail = lab[lab.nli_label == "entailment"]
    display(Markdown(f"**{len(lab)} hand-labelled sentences:** agreement {np.mean(lab.nli_label == lab.your_label):.0%}, "
                     f"Cohen's κ {cohen_kappa_score(lab.your_label, lab.nli_label):.2f}; "
                     f"of the sentences the model calls entailed, {np.mean(entail.your_label == 'entailment'):.0%} "
                     f"are entailed by hand"))
    display(pd.crosstab(lab.your_label, lab.nli_label, rownames=["by hand"], colnames=["NLI model"]))
else:
    print("No hand-labelled sample attached yet: label nli_check_sample_test.csv and attach it before quoting "
          "the attribution metrics.")
print("saved:", sorted(p.name for p in OUT.iterdir()))

## Before reporting these numbers
1. **Run every batch.** Until all `N_BATCHES` batches are in, the results are preliminary (printed in section 3).
2. **Check the cards.** Cards still marked `checked: no` (count printed at the top) must be compared with their
   papers.
3. **Check the NLI model** on the exported sample (section 9). If the hand labels disagree often, report the
   attribution metrics as indicative only.
4. **Report the primary tests as pre-registered**: four metrics, grounded vs. ungrounded, Holm over 12 tests.
   Everything else is secondary. Quote differences with their intervals, not only p-values.
5. **Run-to-run noise.** The pilot showed differences of up to 6 points between two identical runs (temperature 0),
   so treat secondary differences of that size with caution.